# Q4 Sentiment evaluation
Wilson intervals, case-normalized whitespace tokens and exact-token METEOR variant.

In [1]:
from math import sqrt,exp,log
from statistics import mean,stdev
from collections import Counter
mats=[[[100,5,15],[8,80,12],[20,10,50]],[[106,4,10],[6,84,10],[9,7,64]]]
def metrics(m):
    rows=list(map(sum,m)); cols=list(map(sum,zip(*m))); n=sum(rows)
    tp=[m[i][i] for i in range(3)]
    acc=sum(tp)/n
    z=1.959963984540054
    center=(acc+z*z/(2*n))/(1+z*z/n)
    half=z*sqrt(acc*(1-acc)/n+z*z/(4*n*n))/(1+z*z/n)
    per=[(t/c,t/r,2*t/(r+c)) for t,r,c in zip(tp,rows,cols)]
    return acc,(center-half,center+half),per,(mean(x[2] for x in per),sum(r*x[2] for r,x in zip(rows,per))/n,acc)
scores=list(map(metrics,mats))
for x in scores: print(x)
runs=[.847,.840,.853,.837,.850]
print('Ordering mean and sample SD',mean(runs),stdev(runs))
ref='the delivery was late but the product quality is good'.split()
hyp='the product quality is good but delivery was slow'.split()
def grams(words,n): return Counter(zip(*(words[i:] for i in range(n))))
matches=[sum((grams(ref,n)&grams(hyp,n)).values()) for n in range(1,5)]
bp=exp(min(0,1-len(ref)/len(hyp)))
bleu=bp*exp(mean(log((k or .1)/(len(hyp)-i)) for i,k in enumerate(matches)))
def lcs(a,b):
    dp=[[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i,x in enumerate(a,1):
        for j,y in enumerate(b,1):
            dp[i][j]=1+dp[i-1][j-1] if x==y else max(dp[i-1][j],dp[i][j-1])
    return dp[-1][-1]
rouge=[(k/(len(hyp)-n+1),k/(len(ref)-n+1),2*k/(len(hyp)+len(ref)-2*n+2)) for k,n in [(matches[0],1),(matches[1],2),(lcs(ref,hyp),1)]]
# Exact-token METEOR variant. Reverse greedy alignment, then contiguous chunks.
available=list(enumerate(ref))
pairs=[]
for i,x in reversed(list(enumerate(hyp))):
    for j,y in reversed(available):
        if x==y:
            pairs.append((i,j)); available.remove((j,y)); break
pairs.sort()
m=len(pairs)
chunks=1+sum((c-a,d-b)!=(1,1) for (a,b),(c,d) in zip(pairs,pairs[1:]))
p,r=m/len(hyp),m/len(ref)
meteor=(p*r/(.9*p+.1*r))*(1-.5*(chunks/m)**3)
probs=[.60,.40,.50,.80,.70,.30,.25,.50,.20]
nll=-mean(map(log,probs))
ppl=exp(nll)
print('matches, BP, BLEU-4',matches,bp,bleu)
print('ROUGE precision recall F1',rouge)
print('METEOR pairs, chunks, score',pairs,chunks,meteor)
print('Mean NLL and perplexity',nll,ppl)


(0.7666666666666667, (0.715618653578636, 0.8109717617682728), [(0.78125, 0.8333333333333334, 0.8064516129032258), (0.8421052631578947, 0.8, 0.8205128205128205), (0.6493506493506493, 0.625, 0.6369426751592356)], (0.7546357028584273, 0.7659362987080266, 0.7666666666666667))
(0.8466666666666667, (0.8015339429418983, 0.8830335970090831), [(0.8760330578512396, 0.8833333333333333, 0.8796680497925311), (0.8842105263157894, 0.84, 0.8615384615384616), (0.7619047619047619, 0.8, 0.7804878048780488)], (0.8405647720696805, 0.8471767883973126, 0.8466666666666667))
Ordering mean and sample SD 0.8453999999999999 0.006730527468185542
matches, BP, BLEU-4 [8, 5, 3, 2] 0.8948393168143697 0.474954953300978
ROUGE precision recall F1 [(0.8888888888888888, 0.8, 0.8421052631578947), (0.625, 0.5555555555555556, 0.5882352941176471), (0.5555555555555556, 0.5, 0.5263157894736842)]
METEOR pairs, chunks, score [(0, 5), (1, 6), (2, 7), (3, 8), (4, 9), (5, 4), (6, 1), (7, 2)] 3 0.78677398989899
Mean NLL and perplexity